# Lab | LangChain Med

## Objectives

- continue on with lesson 2' example, use different datasets to test what we did in class. Some datasets are suggested in the notebook but feel free to scout other datasets on HuggingFace or Kaggle.
- Find another model on Hugging Face and compare it.
- Modify the prompt to fit your selected dataset.

In [ ]:
import numpy as np
import pandas as pd

## Load the Dataset
As you can see the notebook is ready to work with three different Datasets. Just uncomment the lines of the Dataset you want to use.

I selected Datasets with News. Two of them have just a brief decription of the news, but the other contains the full text.

As we are working in a free and limited space, I limited the number of news to use with the variable MAX_NEWS. Feel free to pull more if you have memory available.

The name of the field containing the text of the new is stored in the variable *DOCUMENT* and the metadata in *TOPIC*

In [ ]:
news = pd.read_csv('/content/articles.csv')
MAX_NEWS = 100
DOCUMENT="Article Body"
TOPIC="Article Header"


ChromaDB requires that the data has a unique identifier. We can make it with this statement, which will create a new column called **Id**.


In [ ]:
news

,Unnamed: 0,Published Date,Author,Source,Article Header,Sub_Headings,Article Body,Url
0,0,"July 7, 2023",Adam Zewe,MIT News Office,Learning the language of molecules to predict ...,This AI system only needs a small amount of da...,['Discovering new materials and drugs typicall...,https://news.mit.edu/2023/learning-language-mo...
1,1,"July 6, 2023",Alex Ouyang,Abdul Latif Jameel Clinic for Machine Learning...,MIT scientists build a system that can generat...,"BioAutoMATED, an open-source, automated machin...",['Is it possible to build machine-learning mod...,https://news.mit.edu/2023/bioautomated-open-so...
2,2,"June 30, 2023",Jennifer Michalowski,McGovern Institute for Brain Research,"When computer vision works more like a brain, ...",Training artificial neural networks with data ...,"['From cameras to self-driving cars, many of t...",https://news.mit.edu/2023/when-computer-vision...
3,3,"June 30, 2023",Mary Beth Gallagher,School of Engineering,Educating national security leaders on artific...,"Experts from MIT’s School of Engineering, Schw...",['Understanding artificial intelligence and ho...,https://news.mit.edu/2023/educating-national-s...
4,4,"June 30, 2023",Adam Zewe,MIT News Office,Researchers teach an AI to write better chart ...,A new dataset can help scientists develop auto...,['Chart captions that explain complex trends a...,https://news.mit.edu/2023/researchers-chart-ca...
...,...,...,...,...,...,...,...,...
1013,1013,"September 13, 1994",NaN,NaN,MIT's Robotic Fish Takes First Swim,NaN,"[""CAMBRIDGE, Mass.--Consider the fish: highly ...",https://news.mit.edu/1994/robotuna
1014,1014,"June 29, 1994","Donna Coveney, News Office",NaN,Can a robot teach us how people learn?,NaN,"['Cog, the newest and most ambitious robot dev...",https://news.mit.edu/1994/robot-0629
1015,1015,"May 18, 1994","Alice C. Waugh, News Office",NaN,Micro-robot holds promise for new surgical tec...,NaN,"['A few years from now, patients with polyps o...",https://news.mit.edu/1994/microrobot-0518
1016,1016,"April 13, 1994","Andrea Cohen, MIT Sea Grant",NaN,Robot journeys beneath arctic ice,NaN,"[""Despite a wind chill index of -80oF and an e...",https://news.mit.edu/1994/robot-0413


In [ ]:
news["id"] = news.index
news.head()

,Unnamed: 0,Published Date,Author,Source,Article Header,Sub_Headings,Article Body,Url,id
0,0,"July 7, 2023",Adam Zewe,MIT News Office,Learning the language of molecules to predict ...,This AI system only needs a small amount of da...,['Discovering new materials and drugs typicall...,https://news.mit.edu/2023/learning-language-mo...,0
1,1,"July 6, 2023",Alex Ouyang,Abdul Latif Jameel Clinic for Machine Learning...,MIT scientists build a system that can generat...,"BioAutoMATED, an open-source, automated machin...",['Is it possible to build machine-learning mod...,https://news.mit.edu/2023/bioautomated-open-so...,1
2,2,"June 30, 2023",Jennifer Michalowski,McGovern Institute for Brain Research,"When computer vision works more like a brain, ...",Training artificial neural networks with data ...,"['From cameras to self-driving cars, many of t...",https://news.mit.edu/2023/when-computer-vision...,2
3,3,"June 30, 2023",Mary Beth Gallagher,School of Engineering,Educating national security leaders on artific...,"Experts from MIT’s School of Engineering, Schw...",['Understanding artificial intelligence and ho...,https://news.mit.edu/2023/educating-national-s...,3
4,4,"June 30, 2023",Adam Zewe,MIT News Office,Researchers teach an AI to write better chart ...,A new dataset can help scientists develop auto...,['Chart captions that explain complex trends a...,https://news.mit.edu/2023/researchers-chart-ca...,4


In [ ]:
#Because it is just a course we select a small portion of News.
subset_news = news.head(MAX_NEWS)

## Import and configure the Vector Database
I'm going to use ChromaDB, the most popular OpenSource embedding Database.

First we need to import ChromaDB, and after that import the **Settings** class from **chromadb.config** module. This class allows us to change the setting for the ChromaDB system, and customize its behavior.

In [ ]:
!pip install chromadb

In [ ]:
import chromadb
from chromadb.config import Settings

Now we need to create the seetings object calling the **Settings** function imported previously. We store the object in the variable **settings_chroma**.

Is necessary to inform two parameters
* chroma_db_impl. Here we specify the database implementation and the format how store the data. I choose ***duckdb***, because his high-performace. It operate primarly in memory. And is fully compatible with SQL. The store format ***parquet*** is good for tabular data. With good compression rates and performance.

* persist_directory: It just contains the directory where the data will be stored. Is possible work without a directory and the data will be stored in memory without persistece, but Kaggle dosn't support that.

In [ ]:
chroma_client = chromadb.PersistentClient(path="/path/to/persist/directory")

## Filling and Querying the ChromaDB Database
The Data in ChromaDB is stored in collections. If the collection exist we need to delete it.

In the next lines, we are creating the collection by calling the ***create_collection*** function in the ***chroma_client*** created above.

In [ ]:
collection_name = "news_collection"
if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
        chroma_client.delete_collection(name=collection_name)

collection = chroma_client.create_collection(name=collection_name)


It's time to add the data to the collection. Using the function ***add*** we need to inform, at least ***documents***, ***metadatas*** and ***ids***.
* In the **document** we store the big text, it's a different column in each Dataset.
* In **metadatas**, we can informa a list of topics.
* In **id** we need to inform an unique identificator for each row. It MUST be unique! I'm creating the ID using the range of MAX_NEWS.


In [ ]:

collection.add(
    documents=subset_news[DOCUMENT].tolist(),
    metadatas=[{TOPIC: topic} for topic in subset_news[TOPIC].tolist()],
    ids=[f"id{x}" for x in range(MAX_NEWS)],
)

In [ ]:
results = collection.query(query_texts=[question], n_results=3)

print(results)

{'ids': [['id85', 'id93', 'id9']], 'embeddings': None, 'documents': [["['As a graduate student doing his master’s thesis on speech recognition at the MIT AI Lab (now the MIT Computer Science and Artificial Intelligence Laboratory), Dan Huttenlocher worked closely with Professor Victor Zue. Well known for pioneering the development of systems that enable an user to interact with computers using spoken language, Zue traveled frequently to Asia — where much of the early research in speech recognition happened during the 1980s. Huttenlocher occasionally accompanied his professor on these trips, many of which involved interactions with members of MIT Industrial Liaison Program, as he recalls. “It was a tremendous opportunity,” according to Huttenlocher, “and it was a large part of what built my interest in engaging with companies and industry in addition to the academic side of research.”', 'Huttenlocher went on to earn his PhD in computer vision at the Institute and has since embarked on a

## Vector MAP

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [ ]:
getado = collection.get(ids="id14",
                       include=["documents", "embeddings"])

In [ ]:
word_vectors = getado["embeddings"]
word_list = getado["documents"]
word_vectors

array([[-8.01109299e-02, -1.72005058e-03,  4.52398648e-03,
         1.41797941e-02,  1.49626136e-02, -3.27144042e-02,
        -9.84603912e-03,  3.17591168e-02, -7.47400448e-02,
         3.45554613e-02, -2.14801170e-02,  9.04189348e-02,
         7.37002194e-02, -3.94032486e-02, -4.25325371e-02,
         1.06930241e-01,  6.60749003e-02,  7.11988360e-02,
        -3.29005234e-02, -2.65482590e-02, -1.82503294e-02,
         2.73044799e-02,  5.00778481e-02,  5.38493730e-02,
        -1.60298035e-01,  4.80691791e-02, -4.93763015e-02,
         1.82914082e-02, -6.53250096e-03, -9.44626257e-02,
         1.04446232e-01,  2.82096863e-03, -4.44842093e-02,
        -1.66042671e-02, -1.99444685e-02,  6.73325434e-02,
        -4.59405109e-02,  6.02066144e-02,  8.10250174e-03,
        -3.68964905e-03, -5.83942235e-02, -6.50444627e-02,
         6.77937567e-02, -4.85466495e-02,  1.08199641e-01,
         1.19970310e-02, -3.60689759e-02, -5.07925116e-02,
         2.31629815e-02, -1.29899755e-02, -1.56283692e-0

Once we have our information inside the Database we can query It, and ask for data that matches our needs. The search is done inside the content of the document, and it dosn't look for the exact word, or phrase. The results will be based on the similarity between the search terms and the content of documents.

The metadata is not used in the search, but they can be utilized for filtering or refining the results after the initial search.


## Loading the model and creating the prompt
TRANSFORMERS!!
Time to use the library **transformers**, the most famous library from [hugging face](https://huggingface.co/) for working with language models.

We are importing:
* **Autotokenizer**: It is a utility class for tokenizing text inputs that are compatible with various pre-trained language models.
* **AutoModelForCasualLLM**: it provides an interface to pre-trained language models specifically designed for language generation tasks using causal language modeling (e.g., GPT models), or the model used in this notebook ***databricks/dolly-v2-3b***.
* **pipeline**: provides a simple interface for performing various natural language processing (NLP) tasks, such as text generation (our case) or text classification.

The model selected is [dolly-v2-3b](https://huggingface.co/databricks/dolly-v2-3b), the smallest Dolly model. It have 3billion paramaters, more than enough for our sample, and works much better than GPT2.

Please, feel free to test [different Models](https://huggingface.co/models?pipeline_tag=text-generation&sort=trending), you need to search for NLP models trained for text-generation. My recomendation is choose "small" models, or we will run out of memory in kaggle.  


In [ ]:
# Update transformers and huggingface_hub libraries to ensure compatibility with Hugging Face models
!pip install --upgrade transformers huggingface_hub

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = "gpt2"    # Dolly-v2 was originally recommended by Databricks, but it could not be loaded in this environment due to availability or compatibility issues, so GPT-2 is used as an alternative.

# It's good practice to add `trust_remote_code=True` when loading models that might have custom code
# or if experiencing issues, although not strictly required for all models.
# For Dolly-v2 models, `trust_remote_code=True` is often recommended.
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
lm_model = AutoModelForCausalLM.from_pretrained(model_id)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

The next step is to initialize the pipeline using the objects created above.

The model's response is limited to 256 tokens, for this project I'm not interested in a longer response, but it can easily be extended to whatever length you want.

Setting ***device_map*** to ***auto*** we are instructing the model to automaticaly select the most appropiate device: CPU or GPU for processing the text generation.  

In [ ]:
pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    device_map="auto",
)

## Creating the extended prompt
To create the prompt we use the result from query the Vector Database  and the sentence introduced by the user.

The prompt have two parts, the **relevant context** that is the information recovered from the database and the **user's question**.

We only need to join the two parts together to create the prompt that we are going to send to the model.

You can limit the lenght of the context passed to the model, because we can get some Memory problems with one of the datasets that contains a realy large text in the document part.

In [ ]:
question = "How is AI being used in healthcare?"
context = str(results["documents"][0][0])[:700]#context = context[0:5120]
results = collection.query(
    query_texts=["AI healthcare medical diagnosis drug development"],
    n_results=1
)
prompt_template = f"""
Answer the question using only the context below.

Relevant context:
{context}

Question:
{question}

Answer:
"""
prompt_template

"\nAnswer the question using only the context below.\n\nRelevant context:\n['As a graduate student doing his master’s thesis on speech recognition at the MIT AI Lab (now the MIT Computer Science and Artificial Intelligence Laboratory), Dan Huttenlocher worked closely with Professor Victor Zue. Well known for pioneering the development of systems that enable an user to interact with computers using spoken language, Zue traveled frequently to Asia — where much of the early research in speech recognition happened during the 1980s. Huttenlocher occasionally accompanied his professor on these trips, many of which involved interactions with members of MIT Industrial Liaison Program, as he recalls. “It was a tremendous opportunity,” according to Huttenlocher, “and it was \n\nQuestion:\nHow is AI being used in healthcare?\n\nAnswer:\n"

Now all that remains is to send the prompt to the model and wait for its response!


In [ ]:
short_prompt = prompt_template[:3000]

lm_response = pipe(
    short_prompt,
    max_new_tokens=50,
    truncation=False
)
print(lm_response[0]["generated_text"])

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer the question using only the context below.

Relevant context:
['As a graduate student doing his master’s thesis on speech recognition at the MIT AI Lab (now the MIT Computer Science and Artificial Intelligence Laboratory), Dan Huttenlocher worked closely with Professor Victor Zue. Well known for pioneering the development of systems that enable an user to interact with computers using spoken language, Zue traveled frequently to Asia — where much of the early research in speech recognition happened during the 1980s. Huttenlocher occasionally accompanied his professor on these trips, many of which involved interactions with members of MIT Industrial Liaison Program, as he recalls. “It was a tremendous opportunity,” according to Huttenlocher, “and it was 

Question:
How is AI being used in healthcare?

Answer:

In a recent interview, Dr. James D. Jones, a graduate student in artificial intelligence at MIT, talked about a different way of working with AI. He used the term "networke